# VOYARA — Validation Deep Learning : embeddings pour la recherche d'activités
**Ons** · Transfer Learning vs Fine-Tuning de `intfloat/multilingual-e5-base`

Tâche : à partir d'une requête libre (« plage calme en famille à Djerba », « nheb nchouf sahra f Tozeur »),
retrouver les bonnes activités du catalogue (115 activités, 10 villes).

Mesuré sur **le même jeu de test (70 requêtes fr / en / ar / darija, jamais utilisé pour entraîner)** :
1. Baseline : TF-IDF (trigrammes à 5-grammes de caractères) + cosinus
2. DL Transfer Learning : e5 **gelé** (embeddings + cosinus)
3. DL Fine-Tuning : e5 **réentraîné** par apprentissage contrastif (MultipleNegativesRankingLoss) sur 291 paires requête ↔ activité

Données (scripts versionnés, rejouables) :
- `services/ai/data/dl/activites.json` ← `services/ai/scripts/dl_recherche_catalogue.py`
- `services/ai/data/eval/recherche_requetes.jsonl` ← `services/ai/scripts/dl_recherche_jeu_test.py`
- `services/ai/data/dl/paires_train.jsonl` ← `services/ai/scripts/dl_recherche_paires.py` (contrôle de fuite du test)

> Local : `.venv` du projet (CPU, fine-tuning ≈ quelques minutes par graine).
> Colab : Exécution → Modifier le type d'exécution → **GPU T4**.

## 0. Environnement et données

In [ ]:
import os, sys
from pathlib import Path

SUR_COLAB = "google.colab" in sys.modules
if SUR_COLAB:
    if not Path("voyara").exists():
        !git clone -q -b ons_work https://github.com/ALI0benjannet/voyara.git
    !pip install -q sentence-transformers scikit-learn
    RACINE = Path("voyara")
else:
    RACINE = Path.cwd() if (Path.cwd() / "services").exists() else Path.cwd().parent
print("Racine du dépôt :", RACINE.resolve())

In [ ]:
import json, random, time, math
import numpy as np, pandas as pd, torch

def fixer_graine(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

SEED = 42; fixer_graine(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Appareil :", DEVICE, "| torch", torch.__version__)

DL_DIR = RACINE / "services/ai/data/dl"
docs = json.load(open(DL_DIR / "activites.json", encoding="utf-8"))          # { titre : texte }
meta = {m["cle"]: m for m in json.load(open(DL_DIR / "activites_meta.json", encoding="utf-8"))}
test = [json.loads(l) for l in open(RACINE / "services/ai/data/eval/recherche_requetes.jsonl", encoding="utf-8")]
paires = [json.loads(l) for l in open(DL_DIR / "paires_train.jsonl", encoding="utf-8")]

ids = list(docs); textes = [docs[i] for i in ids]
print(f"{len(ids)} activités · {len(test)} requêtes de test · {len(paires)} paires d'entraînement")
print("Test par langue :", pd.Series([q["langue"] for q in test]).value_counts().to_dict())
print("Paires par source :", pd.Series([p["source"] for p in paires]).value_counts().to_dict())

In [ ]:
# Règle d'or : aucune requête du test dans l'entraînement (contrôle exact ; le script des paires
# écarte aussi les requêtes proches à ≥ 0,6 de similarité trigrammes).
norm = lambda s: " ".join(s.lower().split())
fuites = {norm(q["requete"]) for q in test} & {norm(p["requete"]) for p in paires}
print("Requêtes du test présentes dans l'entraînement :", len(fuites))
assert not fuites

## 1. Métriques communes (Recall@5, Recall@10, nDCG@10, MRR, latence)

In [ ]:
LANGUES = ["fr", "en", "ar", "darija"]
resultats = []      # une ligne par approche (historique complet, tentatives comprises)
classements = {}    # approche -> rangs par requête (pour l'analyse d'erreurs)

def metriques(rangs_par_requete, requetes):
    res = {"Recall@5": [], "Recall@10": [], "nDCG@10": [], "MRR": []}
    for q, rang in zip(requetes, rangs_par_requete):
        pert = set(q["pertinentes"])
        for k in (5, 10):
            res[f"Recall@{k}"].append(len(pert & set(rang[:k])) / len(pert))
        dcg = sum(1 / np.log2(i + 2) for i, d in enumerate(rang[:10]) if d in pert)
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(pert), 10)))
        res["nDCG@10"].append(dcg / idcg)
        res["MRR"].append(next((1 / (i + 1) for i, d in enumerate(rang) if d in pert), 0))
    return {m: round(float(np.mean(v)), 3) for m, v in res.items()}

def evaluer(nom, scores_fn, entrainement_s=None, garder=True):
    t0 = time.perf_counter()
    rangs = [[ids[j] for j in np.argsort(-scores_fn(q["requete"]))] for q in test]
    latence_ms = (time.perf_counter() - t0) / len(test) * 1000
    ligne = {"approche": nom, **metriques(rangs, test)}
    for lg in LANGUES:
        sel = [i for i, q in enumerate(test) if q["langue"] == lg]
        m = metriques([rangs[i] for i in sel], [test[i] for i in sel])
        ligne[f"nDCG@10_{lg}"] = m["nDCG@10"]; ligne[f"Recall@5_{lg}"] = m["Recall@5"]
    ligne["entrainement_s"] = entrainement_s
    ligne["latence_ms"] = round(latence_ms, 1)
    if garder:
        resultats.append(ligne); classements[nom] = rangs
    print({k: ligne[k] for k in ["approche", "Recall@5", "Recall@10", "nDCG@10", "MRR", "latence_ms"]})
    return ligne

## 2. Baseline — TF-IDF sur les caractères
Référence simple et sans apprentissage profond. Les n-grammes de caractères (3 à 5) tolèrent les fautes
et la darija en lettres latines, mais ne relient pas « colisée » à « amphithéâtre », ni l'arabe au français.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(textes)
X = tfidf.transform(textes)
evaluer("Baseline TF-IDF (caractères)", lambda r: (X @ tfidf.transform([r]).T).toarray().ravel());

## 3. Point 1 — choix du modèle pré-entraîné (mini-comparaison, modèles gelés)
| Modèle | Taille | Pourquoi le tester |
|---|---|---|
| `intfloat/multilingual-e5-base` | 278 M | multilingue (dont l'arabe), **entraîné pour la recherche** (préfixes `query:` / `passage:`), tourne sur CPU |
| `BAAI/bge-m3` | 568 M | souvent un peu meilleur, mais 2 fois plus lourd et plus lent sur CPU |
| `paraphrase-multilingual-MiniLM-L12-v2` | 118 M | très léger, mais entraîné pour la paraphrase, pas pour la recherche |

On garde le meilleur compromis qualité / taille / vitesse (la démo tourne sur un PC portable).

In [ ]:
from sentence_transformers import SentenceTransformer

def scores_modele(model, E, prefixe_requete=""):
    return lambda r: E @ model.encode([f"{prefixe_requete}{r}"], normalize_embeddings=True)[0]

CANDIDATS = [   # (nom, préfixe requête, préfixe passage)
    ("intfloat/multilingual-e5-base", "query: ", "passage: "),
    ("BAAI/bge-m3", "", ""),
    ("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "", ""),
]
comparaison = []
for nom, pq, pp in CANDIDATS:
    t0 = time.perf_counter()
    m = SentenceTransformer(nom, device=DEVICE)
    E_c = m.encode([f"{pp}{t}" for t in textes], normalize_embeddings=True, batch_size=16)
    l = evaluer(f"gelé : {nom.split('/')[-1]}", scores_modele(m, E_c, pq), garder=False)
    comparaison.append({"modele": nom, "parametres_M": round(sum(p.numel() for p in m.parameters()) / 1e6),
                        "nDCG@10": l["nDCG@10"], "Recall@5": l["Recall@5"], "MRR": l["MRR"],
                        "latence_ms": l["latence_ms"], "chargement+index_s": round(time.perf_counter() - t0, 1)})
    del m, E_c
    if DEVICE == "cuda": torch.cuda.empty_cache()
df_comparaison = pd.DataFrame(comparaison); df_comparaison

## 4. Point 2 — Transfer Learning : e5 gelé
Les poids du modèle ne bougent pas : on encode une fois les 115 activités (`passage: …`), puis chaque
requête (`query: …`), et on classe par similarité cosinus.

In [ ]:
MODELE = "intfloat/multilingual-e5-base"
gele = SentenceTransformer(MODELE, device=DEVICE)
E_gele = gele.encode([f"passage: {t}" for t in textes], normalize_embeddings=True, batch_size=16)
evaluer("DL Transfer Learning (e5 gelé)", scores_modele(gele, E_gele, "query: "));

## 5. Point 3 — Fine-Tuning contrastif
**Stratégie** : MultipleNegativesRankingLoss. Pour chaque paire (requête, activité) d'un batch, les autres
activités du batch servent de négatifs : on maximise la similarité de la bonne paire par une entropie croisée
sur la matrice requêtes × activités (échelle 20, comme `losses.MultipleNegativesRankingLoss`).
Boucle écrite à la main (quelques lignes) pour : masquer les **faux négatifs** (même activité deux fois dans
un batch), contrôler les graines et **ne sauvegarder qu'une fois à la fin** (conseil de Boutheina).

**Hyperparamètres** : 3 époques (au-delà, risque de sur-apprentissage avec ~300 paires), lr 2e-5 (standard pour
ce type de modèle), warmup 10 %, batch 16, AdamW. Variante : 6 premières couches gelées.

In [ ]:
import torch.nn.functional as F

ENTRAINER = True        # False : recharge le modèle sauvegardé (démo devant le prof, sans réentraîner)
DOSSIER_MODELE = RACINE / "modeles/e5-voyara-ft"
historique = []         # une ligne par époque et par run (perte moyenne)

def encoder(model, textes_):
    feats = model.tokenize(textes_)
    feats = {k: v.to(model.device) for k, v in feats.items()}
    return F.normalize(model(feats)["sentence_embedding"], dim=-1)

def fine_tuner(graine, couches_gelees=0, epochs=3, lr=2e-5, batch=16, echelle=20.0, nom_run=""):
    fixer_graine(graine)
    model = SentenceTransformer(MODELE, device=DEVICE)
    if couches_gelees:
        tr = model[0].auto_model
        for p in tr.embeddings.parameters(): p.requires_grad = False
        for couche in tr.encoder.layer[:couches_gelees]:
            for p in couche.parameters(): p.requires_grad = False
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr)
    n_batches = math.ceil(len(paires) / batch); total = n_batches * epochs; warmup = int(0.1 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / max(1, warmup) if s < warmup else max(0.0, (total - s) / max(1, total - warmup)))
    model.train(); t0 = time.perf_counter()
    for ep in range(epochs):
        ordre = list(range(len(paires))); random.shuffle(ordre); pertes = []
        for b in range(n_batches):
            lot = [paires[i] for i in ordre[b * batch:(b + 1) * batch]]
            q = encoder(model, [f"query: {p['requete']}" for p in lot])
            d = encoder(model, [f"passage: {docs[p['activite']]}" for p in lot])
            sim = q @ d.T * echelle
            meme = torch.tensor([[a["activite"] == c["activite"] for c in lot] for a in lot], device=sim.device)
            sim = sim.masked_fill(meme & ~torch.eye(len(lot), dtype=torch.bool, device=sim.device), -1e4)
            perte = F.cross_entropy(sim, torch.arange(len(lot), device=sim.device))
            opt.zero_grad(); perte.backward(); opt.step(); sched.step()
            pertes.append(perte.item())
        historique.append({"run": nom_run, "graine": graine, "epoque": ep + 1, "perte": round(float(np.mean(pertes)), 4)})
        print(f"  {nom_run} graine {graine} · époque {ep + 1} : perte {np.mean(pertes):.4f}")
    duree = round(time.perf_counter() - t0, 1)
    model.eval()
    return model, duree

def evaluer_modele(nom, model, duree, garder=True):
    with torch.no_grad():
        E = model.encode([f"passage: {t}" for t in textes], normalize_embeddings=True, batch_size=16)
    return evaluer(nom, scores_modele(model, E, "query: "), entrainement_s=duree, garder=garder)

In [ ]:
# Run principal (graine 42) : modèle sauvegardé une seule fois, à la fin.
if ENTRAINER:
    ft, duree_ft = fine_tuner(42, nom_run="complet")
    ft.save(str(DOSSIER_MODELE))
else:
    ft, duree_ft = SentenceTransformer(str(DOSSIER_MODELE), device=DEVICE), None
evaluer_modele("DL Fine-Tuning (e5, graine 42)", ft, duree_ft);

In [ ]:
# Robustesse : 3 graines (42, 7, 2026) → moyenne ± écart-type (avec peu de données, un même réglage varie).
GRAINES = [42, 7, 2026]
multi = {"complet": [], "6 couches gelées": []}
if ENTRAINER:
    for variante, gel in [("complet", 0), ("6 couches gelées", 6)]:
        for g in GRAINES:
            m_, d_ = fine_tuner(g, couches_gelees=gel, nom_run=variante)
            multi[variante].append(evaluer_modele(f"FT {variante} · graine {g}", m_, d_, garder=False))
            del m_
            if DEVICE == "cuda": torch.cuda.empty_cache()

def resume(lignes, nom):
    cols = ["Recall@5", "Recall@10", "nDCG@10", "MRR", "entrainement_s", "latence_ms"] + \
           [f"nDCG@10_{lg}" for lg in LANGUES] + [f"Recall@5_{lg}" for lg in LANGUES]
    ligne = {"approche": nom}
    for c in cols:
        v = [l[c] for l in lignes]
        ligne[c] = round(float(np.mean(v)), 3)
        if c in ("Recall@5", "Recall@10", "nDCG@10", "MRR"): ligne[f"{c}_ecart_type"] = round(float(np.std(v)), 3)
    return ligne

for variante, lignes in multi.items():
    if lignes: resultats.append(resume(lignes, f"DL Fine-Tuning {variante} (moyenne 3 graines)"))
pd.DataFrame([l for v in multi.values() for l in v])[["approche", "Recall@5", "Recall@10", "nDCG@10", "MRR", "entrainement_s"]] if ENTRAINER else None

## 6. Point 4 — résultats

In [ ]:
df_tout = pd.DataFrame(resultats)
df = df_tout.drop_duplicates(subset=["approche"], keep="last")
DOSSIER_RES = RACINE / "docs/ia/resultats"; DOSSIER_RES.mkdir(parents=True, exist_ok=True)
df_tout.to_csv(DOSSIER_RES / "dl_recherche_historique.csv", index=False)
df.to_csv(DOSSIER_RES / "dl_recherche.csv", index=False)
pd.DataFrame(historique).to_csv(DOSSIER_RES / "dl_recherche_pertes.csv", index=False)
df_comparaison.to_csv(DOSSIER_RES / "dl_recherche_choix_modele.csv", index=False)

glob = ["approche", "Recall@5", "Recall@10", "nDCG@10", "MRR", "entrainement_s", "latence_ms"]
par_langue = ["approche"] + [f"nDCG@10_{lg}" for lg in LANGUES]
with open(DOSSIER_RES / "dl_recherche.md", "w", encoding="utf-8") as f:
    f.write("## Global\n\n" + df[glob].to_markdown(index=False) + "\n\n")
    f.write("## nDCG@10 par langue\n\n" + df[par_langue].to_markdown(index=False) + "\n\n")
    f.write("## Choix du modèle (gelés)\n\n" + df_comparaison.to_markdown(index=False) + "\n")
df[glob]

In [ ]:
df[par_langue]

## 7. Point 5 — analyse critique

In [ ]:
# Requêtes ratées après fine-tuning : aucune activité pertinente dans le top 5.
nom_ft = "DL Fine-Tuning (e5, graine 42)"
lignes = []
for q, r_ft, r_gele, r_tfidf in zip(test, classements[nom_ft], classements["DL Transfer Learning (e5 gelé)"],
                                    classements["Baseline TF-IDF (caractères)"]):
    pert = set(q["pertinentes"])
    rang = lambda r: next((i + 1 for i, d in enumerate(r) if d in pert), None)
    lignes.append({"id": q["id"], "langue": q["langue"], "requete": q["requete"],
                   "rang_tfidf": rang(r_tfidf), "rang_gele": rang(r_gele), "rang_ft": rang(r_ft),
                   "top1_ft": r_ft[0], "attendu": q["pertinentes"][0]})
df_rangs = pd.DataFrame(lignes)
df_rangs.to_csv(DOSSIER_RES / "dl_recherche_rangs.csv", index=False)
print("Ratées après fine-tuning (rien dans le top 5) :", int((df_rangs.rang_ft > 5).sum()), "/", len(df_rangs))
df_rangs[df_rangs.rang_ft > 5]

In [ ]:
# Gains et pertes du fine-tuning par rapport au modèle gelé (rang de la 1re activité pertinente).
df_rangs["gain"] = df_rangs.rang_gele - df_rangs.rang_ft
print("Améliorées :", int((df_rangs.gain > 0).sum()), "· identiques :", int((df_rangs.gain == 0).sum()),
      "· dégradées :", int((df_rangs.gain < 0).sum()))
df_rangs.sort_values("gain")[["id", "langue", "requete", "rang_tfidf", "rang_gele", "rang_ft"]].head(8)

Questions à préparer (voir le rapport `docs/ia/rapports/etape-dl-recherche.md`) :
- Quelles requêtes restent ratées ? Pourquoi (synonymes, nom de ville, requête trop vague) ? 3 exemples.
- Le gain est-il le même en arabe et en darija qu'en français ?
- Les paires sont-elles trop proches des descriptions ? (208 paires sur 291 sont des **titres traduits**.)
- Le gain justifie-t-il le coût (temps d'entraînement, modèle de 1,1 Go à héberger) ?
- Limites : test de 70 requêtes écrit par l'équipe, catalogue de démonstration, pas de vraies requêtes d'utilisateurs.